# 上下文工程

[上下文工程](https://docs.langchain.com/oss/python/langchain/context-engineering)（Context Engineering）对于 Agent 得出正确的结果至关重要。模型回答不好，很多时候不是因为能力不足，而是因为没有获得足以推断出正确结果的上下文信息。通过上下文工程，增强 Agent 获取和管理上下文的能力，是很有必要的。

**LangGraph 将上下文分为三种类型：**

- 模型上下文（Model Context）
- 工具上下文（Tool Context）
- 生命周期上下文（Life-cycle Context）

无论哪种 Context，都需要定义它的 Schema。在这方面，LangGraph 提供了相当高的自由度，你可以使用 `dataclasses`、`pydantic`、`TypedDict` 这些包的任意一个创建你的 Context Schema.

In [13]:
# !pip install ipynbname  # 用于在 Jupyter 中获取当前 notebook 文件路径


In [14]:
import os  # 标准库：操作系统接口
import uuid  # 标准库：生成唯一标识符（用于 thread_id）
import sqlite3  # 标准库：SQLite 数据库接口

from typing import Callable  # 类型注解：可调用对象
from dotenv import load_dotenv  # 从 .env 文件加载环境变量
from dataclasses import dataclass  # 用于定义轻量级数据类（Context Schema）
from langchain_openai import ChatOpenAI  # OpenAI 兼容接口的聊天模型
from langchain.tools import tool, ToolRuntime  # 工具装饰器 + 工具运行时上下文
from langchain.agents import create_agent  # 创建 Agent 的工厂函数
from langchain.agents.middleware import (  # 中间件相关：
    dynamic_prompt,  # @dynamic_prompt 装饰器：动态修改系统提示词
    wrap_model_call,  # @wrap_model_call 装饰器：控制模型调用过程
    ModelRequest,  # 模型请求对象
    ModelResponse,  # 模型响应对象
    SummarizationMiddleware,  # 摘要中间件：压缩过长上下文
)
from langgraph.checkpoint.memory import InMemorySaver  # 内存检查点：保存/恢复 Agent 状态
from langgraph.store.memory import InMemoryStore  # 内存存储：长期记忆
from langgraph.store.sqlite import SqliteStore  # SQLite 存储：持久化长期记忆

# 加载 .env 文件中的环境变量
_ = load_dotenv()

# 加载模型：使用 DashScope（通义千问）的 qwen3-coder-plus 模型
llm = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    model="qwen3-coder-plus",
    temperature=0.7,
)


## 一、动态修改系统提示词

上下文工程与前序章节的中间件（middleware）和记忆（memory）密不可分。上下文的具体实现依赖中间件，而上下文的存储则依赖记忆系统。具体来讲，LangGraph 预置了 `@dynamic_prompt` 中间件，用于动态修改系统提示词。

既然是动态修改，肯定需要某个条件来触发修改。除了开发触发逻辑，我们还需要从智能体中获取触发逻辑所需的即时变量。这些变量通常存储在以下三个存储介质中：

- 运行时（Runtime）- 所有节点共享一个 Runtime。同一时刻，所有节点取到的 Runtime 的值是相同的。一般用于存储时效性要求较高的信息。
- 短期记忆（State）- 在节点之间按顺序传递，每个节点接收上一个节点处理后的 State。主要用于存储 Prompt 和 AI Message。
- 长期记忆（Store）- 负责持久化存储，可以跨 Workflow / Agent 保存信息。可以用来存用户偏好、以前算过的统计值等。

以下三个例子，分别演示如何使用来自 Runtime、State、Store 中的上下文，编写触发条件。

### 1）使用 `State` 管理上下文

利用 `State` 中蕴含的信息操纵 system prompt.

In [15]:
@dynamic_prompt
def state_aware_prompt(request: ModelRequest) -> str:
    """
    基于 State（短期记忆）的动态提示词中间件。
    
    根据对话历史的消息数量，动态调整系统提示词。
    当对话较长时，要求模型更加简洁。
    
    Args:
        request: ModelRequest 对象，包含当前状态和消息列表
    
    Returns:
        拼接后的系统提示词字符串
    """
    # request.messages 是 request.state["messages"] 的快捷方式
    message_count = len(request.messages)

    # 基础提示词
    base = "You are a helpful assistant."

    # 当消息数超过 6 条时，追加简洁要求
    if message_count > 6:
        base += "\nThis is a long conversation - be extra concise."

    # 临时打印 base 看效果
    print(base)

    return base

# 创建 Agent 并注册动态提示词中间件
agent = create_agent(
    model=llm,
    middleware=[state_aware_prompt]
)

# 模拟一段包含 7 条消息的长对话（超过阈值 6）
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "广州今天的天气怎么样？"},
        {"role": "assistant", "content": "广州天气很好"},
        {"role": "user", "content": "吃点什么好呢"},
        {"role": "assistant", "content": "要不要吃香茅鳗鱼煲"},
        {"role": "user", "content": "香茅是什么"},
        {"role": "assistant", "content": "香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉"},
        {"role": "user", "content": "auv 那还等什么，咱吃去吧"},
    ]},
)

# 打印完整消息历史，观察系统提示词的变化

# 每次 AI 要回答之前，都会重新跑一遍 state_aware_prompt，重新生成最新的系统提示词！
for message in result['messages']:
    message.pretty_print()


You are a helpful assistant.
This is a long conversation - be extra concise.
================================ Human Message =================================

广州今天的天气怎么样？
================================== Ai Message ==================================

广州天气很好
================================ Human Message =================================

吃点什么好呢
================================== Ai Message ==================================

要不要吃香茅鳗鱼煲
================================ Human Message =================================

香茅是什么
================================== Ai Message ==================================

香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉
================================ Human Message =================================

auv 那还等什么，咱吃去吧
================================== Ai Message ==================================

走嘞！（ briskly ）


把 `message_count > 6` 里的 6 改成 7，试试看会发生什么。

### 2）使用 `Store` 管理上下文

In [16]:
@dataclass
class Context:
    """Runtime 上下文 Schema：存储用户 ID。"""
    user_id: str

@dynamic_prompt
def store_aware_prompt(request: ModelRequest) -> str:
    """
    基于 Store（长期记忆）的动态提示词中间件。
    
    从 Store 中读取用户偏好，动态调整提示词中的回复风格。
    
    Args:
        request: ModelRequest 对象，包含 runtime 和 store
    
    Returns:
        根据用户偏好拼接的系统提示词
    """
    # 从 Runtime Context 中获取当前用户 ID
    user_id = request.runtime.context.user_id

    # 从 Store 中读取用户偏好（长期记忆）
    # store.get(命名空间, key) 获取存储的值
    store = request.runtime.store
    user_prefs = store.get(("preferences",), user_id)

    base = "You are a helpful assistant."

    # 如果找到了用户偏好，追加到提示词中
    if user_prefs:
        style = user_prefs.value.get("communication_style", "balanced")
        base += f"\nUser prefers {style} responses."

    return base

# 创建内存存储（长期记忆）
store = InMemoryStore()

# 创建 Agent，配置 context_schema 和 store
agent = create_agent(
    model=llm,
    middleware=[store_aware_prompt],  # 注册 Store 感知提示词中间件
    context_schema=Context,  # 指定上下文 Schema 类型
    store=store,  # 传入长期记忆存储
)

# 预置两条用户偏好到 Store 中：
# store.put(命名空间, key, value)
store.put(("preferences",), "user_1", {"communication_style": "Chinese"})  # 用户 1 喜欢中文回复
store.put(("preferences",), "user_2", {"communication_style": "Korean"})   # 用户 2 喜欢韩文回复


In [17]:
# 测试用户 1：喜欢中文回复
result = agent.invoke(
    {"messages": [
        {"role": "system", "content": "You are a helpful assistant. Please be extra concise."},
        {"role": "user", "content": 'What is a "hold short line"?'}
    ]},
    # 传入 Context，指定 user_id = "user_1"
    # Agent 会从 Store 中读取 user_1 的偏好（Chinese）并注入提示词
    context=Context(user_id="user_1"),
)

for message in result['messages']:
    message.pretty_print()


================================ System Message ================================

You are a helpful assistant. Please be extra concise.
================================ Human Message =================================

What is a "hold short line"?
================================== Ai Message ==================================

**保持短横线**（Hold Short Line）是机场跑道上的标记，指示飞机在此处停下等待，不得越过该线进入跑道。

- **作用**：确保飞机在获得塔台许可前不进入活跃跑道
- **位置**：通常位于跑道入口前的滑行道上
- **外观**：由四条黄色平行线组成（两实线+两虚线）

这是航空安全的重要地面标识。


In [18]:
# 测试用户 2：喜欢韩文回复
result = agent.invoke(
    {"messages": [
        {"role": "system", "content": "You are a helpful assistant. Please be extra concise."},
        {"role": "user", "content": 'What is a "hold short line"?'}
    ]},
    # 传入 Context，指定 user_id = "user_2"
    # Agent 会从 Store 中读取 user_2 的偏好（Korean）并注入提示词
    context=Context(user_id="user_2"),
)

for message in result['messages']:
    message.pretty_print()


================================ System Message ================================

You are a helpful assistant. Please be extra concise.
================================ Human Message =================================

What is a "hold short line"?
================================== Ai Message ==================================

"홀드 숏 라인(Hold short line)"은 공항 활주로나 유도로에서 항공기가 대기하도록 지정된 위치 표시선입니다.

한국어로는 **대기선** 또는 **정지선**이라고 합니다.

- 항공기 조종사에게 "Hold short of runway" 등 지시 시 이 선에서 정지해야 함
- 활주로 진입 전 안전 거리 확보 목적
- 일반적으로 노란색 실선으로 표시됨


### 3）使用 `Runtime` 管理上下文

In [19]:
@dataclass
class Context:
    """Runtime 上下文 Schema：用户角色和部署环境。"""
    user_role: str
    deployment_env: str

@dynamic_prompt
def context_aware_prompt(request: ModelRequest) -> str:
    """
    基于 Runtime（运行时上下文）的动态提示词中间件。
    
    根据用户角色（admin/viewer）和部署环境（production/development），
    动态调整系统提示词，控制工具使用权限和安全策略。
    
    Args:
        request: ModelRequest 对象
    
    Returns:
        根据角色和环境拼接的系统提示词
    """
    # 从 Runtime Context 中读取用户角色和部署环境
    user_role = request.runtime.context.user_role
    env = request.runtime.context.deployment_env

    base = "You are a helpful assistant."

    # 根据用户角色控制工具使用权限
    if user_role == "admin":
        base += "\nYou can use the get_weather tool."  # 管理员允许使用天气工具
    else:
        base += "\nYou are prohibited from using the get_weather tool."  # 普通用户禁止使用

    # 生产环境需要格外小心
    if env == "production":
        base += "\nBe extra careful with any data modifications."

    return base

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

# 创建 Agent，注册中间件和工具
agent = create_agent(
    model=llm,
    tools=[get_weather],  # 注册天气工具
    middleware=[context_aware_prompt],  # 注册 Runtime 感知提示词中间件
    context_schema=Context,  # 指定上下文 Schema 类型
    checkpointer=InMemorySaver(),  # 启用检查点，保持会话状态
)


In [20]:
# 测试：user_role = admin，允许使用天气查询工具
config = {'configurable': {'thread_id': str(uuid.uuid4())}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "广州今天的天气怎么样？"}]},
    # 传入 Context：管理员角色 + 生产环境
    # 提示词中将允许使用 get_weather 工具
    context=Context(user_role="admin", deployment_env="production"),
    config=config,
)

for message in result['messages']:
    message.pretty_print()


================================ Human Message =================================

广州今天的天气怎么样？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_7362be0103334aa4af498832)
 Call ID: call_7362be0103334aa4af498832
  Args:
    city: 广州
================================= Tool Message =================================
Name: get_weather

It's always sunny in 广州!
================================== Ai Message ==================================

根据工具返回的信息，广州今天的天气情况是“总是晴天”。不过这可能是一个固定的回复，并不反映实际的天气状况。如需了解实时天气，建议查询权威天气预报平台。


In [21]:
# 测试：user_role = viewer，禁止使用天气查询工具
config = {'configurable': {'thread_id': str(uuid.uuid4())}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "广州今天的天气怎么样？"}]},
    # 传入 Context：普通用户角色 + 生产环境
    # 提示词中将禁止使用 get_weather 工具
    context=Context(user_role="viewer", deployment_env="production"),
    config=config,
)

for message in result['messages']:
    message.pretty_print()


================================ Human Message =================================

广州今天的天气怎么样？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_abc763ed8b944cc298828f25)
 Call ID: call_abc763ed8b944cc298828f25
  Args:
    city: 广州
================================= Tool Message =================================
Name: get_weather

It's always sunny in 广州!
================================== Ai Message ==================================

根据工具返回的信息，广州今天的天气情况是：阳光明媚（sunny）！如果您需要更详细的天气信息，比如温度、湿度等，可以告诉我，我会尽力提供帮助。


In [22]:
result['messages']

[HumanMessage(content='广州今天的天气怎么样？', additional_kwargs={}, response_metadata={}, id='f1f54169-02a3-4906-81be-c759909981ed'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 287, 'total_tokens': 308, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen3-coder-plus', 'system_fingerprint': None, 'id': 'chatcmpl-9ab8af2d-41ed-9f5d-9cd2-0a76d6d5069b', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019de887-6d13-7311-a2b5-b018fba45392-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '广州'}, 'id': 'call_abc763ed8b944cc298828f25', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 287, 'output_tokens': 21, 'total_tokens': 308, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
 ToolMessage(content="It's always sunny in 广州!", name='get_weat

## 二、动态修改消息列表

LangGraph 预制了动态修改消息列表（Messages）的中间件 `@wrap_model_call`。上一节已经演示如何从 `State`、`Store`、`Runtime` 中获取上下文，本节将不再一一演示。在下面这个例子中，我们主要演示如何使用 `Runtime` 将本地文件的内容注入消息列表。

In [23]:
@dataclass
class FileContext:
    """Runtime 上下文 Schema：用户上传的文件列表。"""
    uploaded_files: list[dict]

@wrap_model_call
def inject_file_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """
    文件上下文注入中间件：在模型调用前，将用户上传的文件内容注入到消息列表中。

    使用 @wrap_model_call 装饰器，可以完全控制模型的调用过程。
    这里我们在消息列表末尾追加文件内容，让模型回答时参考这些文件。

    Args:
        request: ModelRequest 对象
        handler: 实际执行模型调用的处理函数

    Returns:
        ModelResponse: 模型返回的响应
    """
    # 从 Runtime Context 中获取用户上传的文件列表
    uploaded_files = request.runtime.context.uploaded_files

    # 获取基础目录路径
    # 获取基础目录：优先使用当前工作目录
    base_dir = os.getcwd()

    file_sections = []
    for file in uploaded_files:
        name, ftype = "", ""
        path = file.get("path")
        if path:
            base_filename = os.path.basename(path)
            stem, ext = os.path.splitext(base_filename)
            name = stem or base_filename
            ftype = (ext.lstrip(".") if ext else None)

            # 构建文件描述内容
            content_list = [f"名称: {name}"]
            if ftype:
                content_list.append(f"类型: {ftype}")

            # 解析相对路径为绝对路径
            abs_path = path if os.path.isabs(path) else os.path.join(base_dir, path)

            # 读取文件内容
            content_block = ""
            if abs_path and os.path.exists(abs_path):
                try:
                    with open(abs_path, "r", encoding="utf-8") as f:
                        content_block = f.read()
                except Exception as e:
                    content_block = f"[读取文件错误 '{abs_path}': {e}]"
            else:
                content_block = "[文件路径缺失或未找到]"

            section = (
                f"---\n"
                f"{chr(10).join(content_list)}\n\n"
                f"{content_block}\n"
                f"---"
            )
            file_sections.append(section)

    # 拼接所有文件内容为一个上下文块（在 for 循环外部）
    file_context = (
        "已加载的会话文件：\n"
        f"{chr(10).join(file_sections)}"
        "\n回答问题时请参考这些文件。"
    )

    # 将文件上下文注入到消息列表末尾
    messages = [
        *request.messages,
        {"role": "user", "content": file_context},
    ]
    # 使用 request.override 替换消息列表
    request = request.override(messages=messages)

    return handler(request)

# 创建 Agent，注册文件注入中间件
agent = create_agent(
    model=llm,
    middleware=[inject_file_context],
    context_schema=FileContext,  # 指定上下文 Schema 为 FileContext
)


In [24]:
# 测试文件注入：提问关于上海地铁规则怪谈文件的问题
result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "关于上海地铁的无脸乘客，有什么需要注意的？",
        }],
    },
    # 传入文件路径，中间件会读取文件内容并注入到消息中
    context=FileContext(uploaded_files=[{"path": "./docs/rule_horror.md"}]),
)

for message in result['messages']:
    message.pretty_print()


IndexError: list index out of range

## 三、在工具中使用上下文

下面，我们尝试在工具中使用存储在 `SqliteStore` 中的上下文信息。

In [ ]:
# 如果 SQLite 数据库文件已存在，先删除它（确保从零开始）
if os.path.exists("user-info.db"):
    os.remove("user-info.db")

# 创建 SQLite 连接，用于持久化存储用户信息
# check_same_thread=False: 允许多线程访问
# isolation_level=None: 自动提交模式
conn = sqlite3.connect("user-info.db", check_same_thread=False, isolation_level=None)
# 启用 WAL（Write-Ahead Logging）模式：提高并发读写性能
conn.execute("PRAGMA journal_mode=WAL;")
# 设置忙等待超时时间为 30 秒
conn.execute("PRAGMA busy_timeout = 30000;")

# 创建 SqliteStore 实例
store = SqliteStore(conn)

# 预置两条用户信息到 Store 中：
# store.put(命名空间, key, value)
store.put(("user_info",), "柳如烟", {"description": "清冷才女，身怀绝技，为寻身世之谜踏入江湖。", "birthplace": "吴兴县"})
store.put(("user_info",), "苏慕白", {"description": "孤傲剑客，剑法超群，背负家族血仇，隐于市井追寻真相。", "birthplace": "杭县"})


### 1）基础用例

使用 `ToolRuntime`

In [ ]:
@tool
def fetch_user_data(
    user_id: str,
    runtime: ToolRuntime  # 工具运行时上下文，由框架自动注入
) -> str:
    """
    从 Store 中获取用户信息的工具函数。
    
    使用 ToolRuntime 访问 Store（长期记忆），根据 user_id 查找用户描述。
    
    Args:
        user_id: 用户的唯一标识符（如 "柳如烟"）
        runtime: 工具运行时上下文，提供 store 等运行时资源
    
    Returns:
        用户的描述字符串，如果未找到则返回空字符串
    """
    # 从 runtime 中获取 Store
    store = runtime.store
    # 从 Store 中查找用户信息：store.get(命名空间, key)
    user_info = store.get(("user_info",), user_id)

    user_desc = ""
    if user_info:
        user_desc = user_info.value.get("description", "")

    return user_desc

# 创建 Agent，注册工具和 Store
agent = create_agent(
    model=llm,
    tools=[fetch_user_data],
    store=store,  # 传入 SQLite 持久化存储
)


In [ ]:
# 测试：查询 "柳如烟" 的用户信息
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "五分钟之内，我要柳如烟的全部信息"
    }]
})

for message in result['messages']:
    message.pretty_print()


### 2）复杂一点的例子

使用 `ToolRuntime[Context]`

In [ ]:
@dataclass
class Context:
    """Runtime 上下文 Schema：指定要查询的字段 key。"""
    key: str

@tool
def fetch_user_data(
    user_id: str,
    runtime: ToolRuntime[Context]  # 带 Context 类型参数的工具运行时
) -> str:
    """
    从 Store 中获取用户指定字段信息的工具函数。
    
    使用 ToolRuntime[Context] 访问运行时上下文，
    根据 context.key 决定返回用户的哪个属性（description / birthplace）。
    
    Args:
        user_id: 用户的唯一标识符
        runtime: 带 Context 的工具运行时，可访问 runtime.context 和 runtime.store
    
    Returns:
        格式化的字段值字符串，如 "birthplace: 吴兴县"
    """
    # 从 Runtime Context 中获取要查询的字段名
    key = runtime.context.key

    store = runtime.store
    user_info = store.get(("user_info",), user_id)

    user_desc = ""
    if user_info:
        # 根据 context.key 动态选择返回哪个字段
        user_desc = user_info.value.get(key, "")

    return f"{key}: {user_desc}"

# 创建 Agent
agent = create_agent(
    model=llm,
    tools=[fetch_user_data],
    store=store,
)


In [ ]:
# 测试：查询 "柳如烟" 的出生地（birthplace）
result = agent.invoke(
    {"messages": [{"role": "user", "content": "五分钟之内，我要柳如烟的全部信息"}]},
    # 传入 Context，指定 key = "birthplace"
    # 工具会返回该字段的值而非 description
    context=Context(key="birthplace"),
)

for message in result['messages']:
    message.pretty_print()


### 四、压缩上下文

LangChain 提供了内置的中间件 `SummarizationMiddleware` 用于压缩上下文。该中间件维护的是典型的 **生命周期上下文**，与 **模型上下文** 和 **工具上下文** 的瞬态更新不同，生命周期上下文会持续更新：持续将旧消息替换为摘要。

除非上下文超长，导致模型能力降低，否则不需要使用 `SummarizationMiddleware`。一般来说，触发摘要的值可以设得较大。比如：

- `max_tokens_before_summary`: 3000
- `messages_to_keep`: 20

> 如果你想了解更多关于上下文腐坏（Context Rot）的信息，Chroma 团队在 2025 年 7 月 14 日发布的 [*Context Rot: How Increasing Input Tokens Impacts LLM Performance*](https://research.trychroma.com/context-rot)，系统性地揭示了长上下文导致模型性能退化的现象。

In [ ]:
# 创建短期记忆检查点：用于保存 Agent 会话状态
checkpointer = InMemorySaver()

# 创建带内置摘要中间件的 Agent
# SummarizationMiddleware 会在上下文过长时自动将旧消息压缩为摘要
# 为了让示例中快速触发摘要，这里的触发值设得很小（生产环境应设大一些）
agent = create_agent(
    model=llm,
    middleware=[
        SummarizationMiddleware(
            model=llm,  # 用于生成摘要的模型
            trigger=('tokens', 40),  # 当 token 数达到 40 时触发摘要生成
            keep=('messages', 1),    # 摘要后保留最后 1 条消息
        ),
    ],
)


In [ ]:
# 测试：发送一段 7 条消息的对话，触发上下文压缩
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "广州今天的天气怎么样？"},
        {"role": "assistant", "content": "广州天气很好"},
        {"role": "user", "content": "吃点什么好呢"},
        {"role": "assistant", "content": "要不要吃香茅鳗鱼煲"},
        {"role": "user", "content": "香茅是什么"},
        {"role": "assistant", "content": "香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉"},
        {"role": "user", "content": "auv 那还等什么，咱吃去吧"},
    ]},
    checkpointer=checkpointer,
)

for message in result['messages']:
    message.pretty_print()
